In [ ]:
import numpy as np
anh = np.zeros((4000, 3000, 3))   # ảnh 4K — ~100MB

# Nếu slicing tạo copy:
vung_roi = anh[100:200, 100:200]  # tốn thêm ~3MB mỗi lần cắt
# Cắt 100 lần = 300MB bộ nhớ thêm — chỉ để "nhìn" vào dữ liệu
print(anh)

[[[0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  ...
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]]

 [[0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  ...
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]]

 [[0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  ...
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]]

 ...

 [[0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  ...
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]]

 [[0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  ...
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]]

 [[0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  ...
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]]]


In [ ]:
import numpy as np

def clean_data(data):
    d = data.copy()

    # Bước 1: clip âm về 0
    d = d.clip(min=0)

    # Bước 2: Min-Max theo từng cột
    col_min = d.min(axis=0)          # shape (3,)
    col_max = d.max(axis=0)          # shape (3,)
    d = (d - col_min) / (col_max - col_min)   # broadcasting

    return d

data = np.random.randn(5, 3)   # có giá trị âm
ket_qua = clean_data(data)

# Kiểm tra
print(data)      # phải giống hệt trước khi gọi hàm
print(ket_qua)   # giá trị trong [0, 1], không có âm

[[ 0.298001    0.42853872 -0.2300368 ]
 [-1.86088466  1.57858429  0.20812331]
 [-0.1386404  -1.27707449  0.69800614]
 [-0.30387304 -2.00729042 -1.64752303]
 [ 0.57324575 -0.48880974  0.1892957 ]]
[[0.5198486  0.27147028 0.        ]
 [0.         1.         0.29816831]
 [0.         0.         1.        ]
 [0.         0.         0.        ]
 [1.         0.         0.27119489]]


In [ ]:
import numpy as np
import time

def tinh_diem_rui_ro_nhanh(diem, thu_nhap, du_no):
    dieu_kien = [diem >= 750, diem >= 650]
    he_so     = [0.1,         0.3        ]
    r   = np.select(dieu_kien, he_so, default=0.6)  # ← gán vào r
    dti = du_no / (thu_nhap * 12)
    return r * (1 + dti)

def tinh_diem_rui_ro_cham(diem, thu_nhap, du_no):
    ket_qua = []
    for i in range(len(diem)):
        d = diem[i]
        t = thu_nhap[i]
        n = du_no[i]
        if d >= 750:
            r = 0.1
        elif d >= 650:
            r = 0.3
        else:
            r = 0.6
        dti = n / (t * 12)
        ket_qua.append(r * (1 + dti))
    return np.array(ket_qua)


np.random.seed(42)
n = 1_000_000
diem      = np.random.normal(680, 80, n).clip(300, 850)
thu_nhap  = np.random.lognormal(3, 0.5, n) * 1e6
du_no     = np.random.uniform(0, 500e6, n)

# Đo hàm chậm
t0 = time.time()
kq_cham = tinh_diem_rui_ro_cham(diem, thu_nhap, du_no)
t_cham  = time.time() - t0

# Đo hàm nhanh
t0 = time.time()
kq_nhanh = tinh_diem_rui_ro_nhanh(diem, thu_nhap, du_no)
t_nhanh  = time.time() - t0

print(f"Chậm : {t_cham:.3f}s")
print(f"Nhanh: {t_nhanh:.4f}s")
print(f"Nhanh hơn: {t_cham/t_nhanh:.0f}x")
print(f"Kết quả khớp: {np.allclose(kq_cham, kq_nhanh)}")


Chậm : 0.752s
Nhanh: 0.0137s
Nhanh hơn: 55x
Kết quả khớp: True


In [ ]:
'''BÀI TẬP
Input : ma trận (n, 5) — đã chuẩn hóa Z-score
Output: vector n điểm trong [300, 850]

Yêu cầu:
1. Dot product với trọng số [0.3, 0.25, 0.2, 0.15, 0.1]
2. Scale kết quả về [300, 850] bằng Min-Max
3. Phân loại bằng np.select:
     >= 750 → "Excellent"
     >= 650 → "Good"
     >= 550 → "Fair"
     còn lại → "Poor"
4. Không có side effects
5. Test với n=1,000,000 — đo thời gian'''

import numpy as np

def credit_score(features):
    #1. Dot product với trọng số [0.3, 0.25, 0.2, 0.15, 0.1]
    weight = np.array([0.3, 0.25, 0.2, 0.15, 0.1])
    dp = features @ weight
    #2. Scale kết quả về [300, 850] bằng Min-Max
    dp_min = dp.clip(min = 300)
    dp_max = dp.clip(max = 850)
    col_dp_min = dp_min.min(axis = 0)
    col_dp_max = dp_max.max(axis = 0)
    dp = (dp - col_dp_min)/(col_dp_max - col_dp_min)
    #3. Phân loại
    diem = [dp >= 750, dp >= 650, dp >= 550]
    rank = ["Excellent", "Good", "Fair"]
    result = np.select(diem, rank, default = "Poor")
    return dp, result



In [ ]:
import numpy as np
import time

def credit_score(features):
    # 1. Dot product với trọng số [0.3, 0.25, 0.2, 0.15, 0.1]
    # (Vì features ĐÃ là Z-score, không cần tính lại trung bình và độ lệch chuẩn)
    weights = np.array([0.3, 0.25, 0.2, 0.15, 0.1])
    raw_scores = features @ weights

    # 2. Scale kết quả về [300, 850] bằng Min-Max
    min_val = np.min(raw_scores)
    max_val = np.max(raw_scores)

    # Áp dụng công thức scale
    dp = 300 + (raw_scores - min_val) * (850 - 300) / (max_val - min_val)

    # 3. Phân loại bằng np.select
    conditions = [
        dp >= 750,
        dp >= 650,
        dp >= 550
    ]
    choices = ["Excellent", "Good", "Fair"]
    result = np.select(conditions, choices, default="Poor")

    # 4. Không có side effects (features gốc không bị thay đổi)
    return dp, result

# ==========================================
# 5. Test với n=1,000,000 — đo thời gian
# ==========================================
if __name__ == "__main__":
    n = 1_000_000

    # Tạo ma trận giả lập n x 5, phân phối chuẩn (đại diện cho dữ liệu đã chuẩn hóa Z-score)
    np.random.seed(42)
    test_features = np.random.randn(n, 5)

    print(f"Đang xử lý {n:,} bản ghi...\n")

    # Bắt đầu đo thời gian
    start_time = time.time()
    scores, categories = credit_score(test_features)
    end_time = time.time()

    # In kết quả
    print(f"Thời gian thực thi: {end_time - start_time:.4f} giây")
    print("-" * 30)
    print("Top 5 điểm số đầu tiên:")
    for i in range(5):
        print(f"Điểm: {scores[i]:.2f} - Xếp loại: {categories[i]}")

    # Kiểm tra nhanh max/min để đảm bảo code scale đúng
    print("-" * 30)
    print(f"Min score: {np.min(scores):.2f} (Kỳ vọng: 300.00)")
    print(f"Max score: {np.max(scores):.2f} (Kỳ vọng: 850.00)")

Đang xử lý 1,000,000 bản ghi...

Thời gian thực thi: 0.0948 giây
------------------------------
Top 5 điểm số đầu tiên:
Điểm: 639.66 - Xếp loại: Fair
Điểm: 641.24 - Xếp loại: Fair
Điểm: 502.42 - Xếp loại: Poor
Điểm: 506.12 - Xếp loại: Poor
Điểm: 600.24 - Xếp loại: Fair
------------------------------
Min score: 300.00 (Kỳ vọng: 300.00)
Max score: 850.00 (Kỳ vọng: 850.00)
